# Notebook 02 — Preprocessing Data (Gabungan, per Tema)
## Early Warning System Krisis Pariwisata Bali

Notebook ini menggabungkan **02_1 (Preprocessing Timeseries)** dan **02_2 (Added Preprocessing — Data Eksternal)**
menjadi satu notebook yang disusun **per tema**, bukan sekadar ditempel berurutan. Tujuannya: alur dari data mentah
ke setiap komponen `crisis_score` (tourism / economy / sentiment / external risk) menjadi terlihat jelas.

**Struktur:**
0. Setup
1. Tema: Pariwisata
2. Tema: Ekonomi & Moneter
3. Tema: Akomodasi
4. Tema: Risiko Eksternal / Bencana
5. Tema: Sinyal Digital & Media
6. Validasi Gabungan & Merge Final
7. Save

**Catatan perbaikan dari notebook asli:**
- Bug f-string `{{ }}` di semua cell validasi (02_1) yang membuat output print **literal teks**, bukan nilai
  variabel — sudah diperbaiki jadi `{ }` tunggal di semua bagian.
- Duplikasi `inflasi_df.to_csv(...)` (tersimpan 2x dengan kode identik) — disederhanakan jadi satu kali save.
- Tidak ada cell yang men-generate atau menyimpan gambar/plot ke `data/processed/` (sudah dicek di kedua notebook asal) — bagian ini sengaja tetap murni preprocessing tanpa visualisasi.
- **Ditambahkan Section 3.2** Lama Menginap Hotel Bintang & Non Bintang — dataset ini sebelumnya
  di-load di NB01 tapi tidak pernah diproses. Lama menginap adalah *leading indicator* penurunan
  kualitas kunjungan (sebelum angka wisman turun) — berguna untuk `crisis_component_tourism`.


---
## 0. Setup

In [1]:
import os
import numpy as np
import pandas as pd
from math import radians, sin, cos, sqrt, atan2
import warnings
warnings.filterwarnings('ignore')

print('✓ Library siap')


✓ Library siap


---
## 1. Tema: Pariwisata

Mencakup jumlah wisatawan mancanegara (wisman), wisatawan domestik (wisnus), dan porsi wisman Bali
terhadap total Indonesia (`bali_share_pct`). Tiga sinyal ini adalah komponen utama `crisis_component_tourism`.


### 1.1 Wisman Gabungan — `Gab_Data_Wisman_Bali.xlsx`
Dataset sudah bersih (Tanggal, Banyak), hanya perlu rename dan convert tipe data.

In [2]:
# Load
wisman = pd.read_excel('../data/raw/Gab_Data_Wisman_Bali.xlsx')

# Rename kolom
wisman.columns = ['date', 'wisman']

# Convert datetime
wisman['date'] = pd.to_datetime(wisman['date'])

# Sort ascending
wisman = wisman.sort_values('date').reset_index(drop=True)

# Buat kolom bulan (periode)
wisman['month'] = wisman['date'].dt.to_period('M')

print('Missing values:', wisman.isnull().sum().to_dict())
print('Rentang data:', wisman['date'].min(), 'sampai', wisman['date'].max())
print('Total baris:', len(wisman))
print()
wisman.head(5)


Missing values: {'date': 0, 'wisman': 0, 'month': 0}
Rentang data: 2009-01-01 00:00:00 sampai 2026-04-01 00:00:00
Total baris: 208



,date,wisman,month
0,2009-01-01,174541,2009-01
1,2009-02-01,147704,2009-02
2,2009-03-01,168205,2009-03
3,2009-04-01,188776,2009-04
4,2009-05-01,190803,2009-05


In [3]:
wisman.to_csv('../data/processed/wisman_clean.csv', index=False)
print('✓ wisman_clean.csv disimpan')


✓ wisman_clean.csv disimpan


In [4]:
# ── Validation: Wisman Gabungan ─────────────────────────────────
print('=== VALIDATION: Wisman Gabungan ===')
print(f'Shape         : {wisman.shape}')
print(f'Rentang       : {wisman["month"].min()} -> {wisman["month"].max()}')
print(f'Missing       :\n{wisman.isnull().sum().to_string()}')
print(f'Duplikat month: {wisman["month"].duplicated().sum()}')
print()
print('Preview:')
print(wisman.head(3).to_string())


=== VALIDATION: Wisman Gabungan ===
Shape         : (208, 3)
Rentang       : 2009-01 -> 2026-04
Missing       :
date      0
wisman    0
month     0
Duplikat month: 0

Preview:
        date  wisman    month
0 2009-01-01  174541  2009-01
1 2009-02-01  147704  2009-02
2 2009-03-01  168205  2009-03


### 1.2 Wisatawan Domestik Bulanan 2004–2025
Format BPS wide (baris=bulan, kolom=tahun) → di-melt jadi format panjang.

In [5]:
path = '../data/raw/wisnus_bali_2004_2025.xlsx'
sheet_names = ['2004-2012', '2013-2021', '2022-2030']

bulan_map_wisnus = {
    'Januari': '01', 'Pebruari': '02', 'Maret': '03', 'April': '04',
    'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
    'September': '09', 'Oktober': '10', 'Nopember': '11', 'Desember': '12',
}

frames = []
for sheet in sheet_names:
    df = pd.read_excel(path, sheet_name=sheet, skiprows=3, header=0)
    bulan_col = df.columns[0]
    tahun_cols = [c for c in df.columns[1:] if str(c) not in ('nan', 'None', 'TOTAL')]

    df[bulan_col] = df[bulan_col].astype(str).str.strip()
    df = df[df[bulan_col].isin(bulan_map_wisnus.keys())].copy()

    melted = df.melt(
        id_vars=[bulan_col],
        value_vars=tahun_cols,
        var_name='tahun',
        value_name='wisnus'
    ).rename(columns={bulan_col: 'bulan_nama'})

    frames.append(melted)
    print(f"Sheet '{sheet}': {len(melted)} rows")

df_all = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df_all)} rows")


Sheet '2004-2012': 108 rows
Sheet '2013-2021': 108 rows
Sheet '2022-2030': 108 rows

Total: 324 rows


In [6]:
df_all['bulan_num'] = df_all['bulan_nama'].map(bulan_map_wisnus)
df_all['tahun']      = df_all['tahun'].astype(str).str[:4]
df_all['wisnus']     = pd.to_numeric(df_all['wisnus'], errors='coerce')
df_all['date']       = pd.to_datetime(
    df_all['tahun'] + '-' + df_all['bulan_num'] + '-01',
    errors='coerce'
)

wisnus_long = (
    df_all[['date', 'wisnus']]
    .dropna()
    .sort_values('date')
    .reset_index(drop=True)
)
wisnus_long.insert(1, 'month', wisnus_long['date'].dt.to_period('M'))

print(f"Shape  : {wisnus_long.shape}")
print(f"Rentang: {wisnus_long['date'].min().date()} -> {wisnus_long['date'].max().date()}")
print(f"Kolom  : {wisnus_long.columns.tolist()}")
wisnus_long.head(12)


Shape  : (264, 3)
Rentang: 2004-01-01 -> 2025-12-01
Kolom  : ['date', 'month', 'wisnus']


,date,month,wisnus
0,2004-01-01,2004-01,167106.0
1,2004-02-01,2004-02,133660.0
2,2004-03-01,2004-03,118369.0
3,2004-04-01,2004-04,129730.0
4,2004-05-01,2004-05,142186.0
5,2004-06-01,2004-06,167718.0
6,2004-07-01,2004-07,212463.0
7,2004-08-01,2004-08,171034.0
8,2004-09-01,2004-09,168420.0
9,2004-10-01,2004-10,150827.0


In [7]:
wisnus_long.to_csv('../data/processed/wisnus_clean.csv', index=False)
print('✓ wisnus_clean.csv disimpan')


✓ wisnus_clean.csv disimpan


In [8]:
# ── Validation: Wisnus Domestik ─────────────────────────────────
print('=== VALIDATION: Wisnus Domestik ===')
print(f'Shape         : {wisnus_long.shape}')
print(f'Rentang       : {wisnus_long["month"].min()} -> {wisnus_long["month"].max()}')
print(f'Missing       :\n{wisnus_long.isnull().sum().to_string()}')
print(f'Duplikat month: {wisnus_long["month"].duplicated().sum()}')
print()
print('Preview:')
print(wisnus_long.head(3).to_string())


=== VALIDATION: Wisnus Domestik ===
Shape         : (264, 3)
Rentang       : 2004-01 -> 2025-12
Missing       :
date      0
month     0
wisnus    0
Duplikat month: 0

Preview:
        date    month    wisnus
0 2004-01-01  2004-01  167106.0
1 2004-02-01  2004-02  133660.0
2 2004-03-01  2004-03  118369.0


### 1.3 Wisman Bali vs Indonesia Tahunan (1969–2025)
Dua tabel side-by-side dalam satu sheet. Dipakai untuk menghitung `bali_share_pct`.

In [9]:
wisman_indo_raw = pd.read_excel(
    '../data/raw/banyaknya-wisatawan-mancanegara-ke-bali-dan-indonesia.xls',
    engine='xlrd',
    header=None
)

def parse_half_table(df, col_start, col_end):
    """Parse satu sisi tabel wisman Bali vs Indonesia."""
    sub = df.iloc[5:, col_start:col_end+1].copy()
    sub.columns = ['tahun', 'indonesia_total', 'indonesia_growth', 'bali_total', 'bali_growth']
    sub = sub.dropna(subset=['tahun'])
    sub['tahun'] = pd.to_numeric(sub['tahun'], errors='coerce')
    sub = sub.dropna(subset=['tahun'])
    sub['tahun'] = sub['tahun'].astype(int)
    sub['indonesia_total'] = pd.to_numeric(sub['indonesia_total'], errors='coerce')
    sub['bali_total'] = pd.to_numeric(sub['bali_total'], errors='coerce')
    return sub[['tahun', 'indonesia_total', 'bali_total']]

left  = parse_half_table(wisman_indo_raw, 0, 4)
right = parse_half_table(wisman_indo_raw, 6, 10)
wisman_indo_annual = pd.concat([left, right], ignore_index=True)
wisman_indo_annual = wisman_indo_annual.sort_values('tahun').reset_index(drop=True)

wisman_indo_annual['bali_share_pct'] = (
    wisman_indo_annual['bali_total'] / wisman_indo_annual['indonesia_total'] * 100
).round(2)

print('Shape:', wisman_indo_annual.shape)
print('Rentang tahun:', wisman_indo_annual['tahun'].min(), '-', wisman_indo_annual['tahun'].max())
wisman_indo_annual.tail(10)


Shape: (57, 4)
Rentang tahun: 1969 - 2025


,tahun,indonesia_total,bali_total,bali_share_pct
47,2016,11519275,4927937,42.78
48,2017,14039799,5697739,40.58
49,2018,15806191,6070473,38.41
50,2019,16106954,6275210,38.96
51,2020,4052923,1069473,26.39
52,2021,1557530,51,0.00
53,2022,5889031,2155747,36.61
54,2023,11677825,5273258,45.16
55,2024,13902420,6333360,45.56
56,2025,15386646,6948754,45.16


In [10]:
wisman_indo_annual.to_csv('../data/processed/wisman_vs_indonesia_clean.csv', index=False)
print('✓ wisman_vs_indonesia_clean.csv disimpan')
print('Kolom:', wisman_indo_annual.columns.tolist())
print('Catatan: Data tahunan — akan di-join ke timeline bulanan berdasarkan tahun di NB selanjutnya')


✓ wisman_vs_indonesia_clean.csv disimpan
Kolom: ['tahun', 'indonesia_total', 'bali_total', 'bali_share_pct']
Catatan: Data tahunan — akan di-join ke timeline bulanan berdasarkan tahun di NB selanjutnya


In [11]:
# ── Validation: Wisman Bali vs Indonesia ─────────────────────────────────
print('=== VALIDATION: Wisman Bali vs Indonesia ===')
print(f'Shape         : {wisman_indo_annual.shape}')
print(f'Rentang       : {wisman_indo_annual["tahun"].min()} -> {wisman_indo_annual["tahun"].max()}')
print(f'Missing       :\n{wisman_indo_annual.isnull().sum().to_string()}')
print(f'Duplikat tahun: {wisman_indo_annual["tahun"].duplicated().sum()}')
print()
print('Preview:')
print(wisman_indo_annual.head(3).to_string())


=== VALIDATION: Wisman Bali vs Indonesia ===
Shape         : (57, 4)
Rentang       : 1969 -> 2025
Missing       :
tahun              0
indonesia_total    0
bali_total         0
bali_share_pct     0
Duplikat tahun: 0

Preview:
   tahun  indonesia_total  bali_total  bali_share_pct
0   1969            86067       11278           13.10
1   1970           129319       24340           18.82
2   1971           178781       34313           19.19


---
## 2. Tema: Ekonomi & Moneter

Kurs USD/IDR, kurs harian multi-currency, inflasi, dan indeks ekonomi global tertimbang (World Bank).
Komponen ini menyusun `crisis_component_economy`.


### 2.1 Kurs USD/IDR Historical

In [12]:
usd = pd.read_csv('../data/raw/USD_IDR Historical Data.csv')
usd = usd[['Date', 'Price']].copy()
usd.columns = ['date', 'usd_idr']

usd['date'] = pd.to_datetime(usd['date'], format='%m/%d/%Y')
usd['usd_idr'] = usd['usd_idr'].astype(str).str.replace(',', '')
usd['usd_idr'] = pd.to_numeric(usd['usd_idr'], errors='coerce')
usd = usd.sort_values('date').reset_index(drop=True)

usd['month'] = usd['date'].dt.to_period('M')
monthly_usd = usd.groupby('month')['usd_idr'].mean().reset_index()
monthly_usd.columns = ['month', 'usd_idr_avg']

print('Shape harian :', usd.shape)
print('Shape bulanan:', monthly_usd.shape)
print('Rentang      :', usd['date'].min(), '->', usd['date'].max())
monthly_usd.tail(5)


Shape harian : (3858, 3)
Shape bulanan: (180, 2)
Rentang      : 2010-01-01 00:00:00 -> 2024-12-31 00:00:00


,month,usd_idr_avg
175,2024-08,15734.772727
176,2024-09,15318.480952
177,2024-10,15557.608696
178,2024-11,15813.095238
179,2024-12,16035.681818


In [13]:
monthly_usd.to_csv('../data/processed/monthly_usd.csv', index=False)
print('✓ monthly_usd.csv disimpan')


✓ monthly_usd.csv disimpan


In [14]:
# ── Validation: Kurs USD/IDR Bulanan ─────────────────────────────────
print('=== VALIDATION: Kurs USD/IDR Bulanan ===')
print(f'Shape         : {monthly_usd.shape}')
print(f'Rentang       : {monthly_usd["month"].min()} -> {monthly_usd["month"].max()}')
print(f'Missing       :\n{monthly_usd.isnull().sum().to_string()}')
print(f'Duplikat month: {monthly_usd["month"].duplicated().sum()}')
print()
print('Preview:')
print(monthly_usd.head(3).to_string())


=== VALIDATION: Kurs USD/IDR Bulanan ===
Shape         : (180, 2)
Rentang       : 2010-01 -> 2024-12
Missing       :
month          0
usd_idr_avg    0
Duplikat month: 0

Preview:
     month  usd_idr_avg
0  2010-01  9275.357143
1  2010-02  9336.725000
2  2010-03  9166.804348


### 2.2 Daily Forex Rates (Filter IDR)

In [15]:
forex = pd.read_csv('../data/raw/daily_forex_rates.csv')
forex_idr = forex[forex['currency'] == 'IDR'].copy()
forex_idr = forex_idr[['date', 'exchange_rate']].copy()

forex_idr['date'] = pd.to_datetime(forex_idr['date'])
forex_idr = forex_idr.sort_values('date').reset_index(drop=True)

forex_idr['month'] = forex_idr['date'].dt.to_period('M')
monthly_forex = forex_idr.groupby('month')['exchange_rate'].mean().reset_index()
monthly_forex.columns = ['month', 'idr_eur_rate']

print('Shape harian :', forex_idr.shape)
print('Shape bulanan:', monthly_forex.shape)
monthly_forex.tail(5)


Shape harian : (3263, 3)
Shape bulanan: (139, 2)


,month,idr_eur_rate
134,2026-01,19729.961342
135,2026-02,19903.853782
136,2026-03,19595.017187
137,2026-04,20026.122880
138,2026-05,20459.534496


In [16]:
monthly_forex.to_csv('../data/processed/monthly_forex.csv', index=False)
print('✓ monthly_forex.csv disimpan')


✓ monthly_forex.csv disimpan


In [17]:
# ── Validation: Forex Daily -> Bulanan ─────────────────────────────────
print('=== VALIDATION: Forex Daily -> Bulanan ===')
print(f'Shape         : {monthly_forex.shape}')
print(f'Rentang       : {monthly_forex["month"].min()} -> {monthly_forex["month"].max()}')
print(f'Missing       :\n{monthly_forex.isnull().sum().to_string()}')
print(f'Duplikat month: {monthly_forex["month"].duplicated().sum()}')
print()
print('Preview:')
print(monthly_forex.head(3).to_string())


=== VALIDATION: Forex Daily -> Bulanan ===
Shape         : (139, 2)
Rentang       : 2014-11 -> 2026-05
Missing       :
month           0
idr_eur_rate    0
Duplikat month: 0

Preview:
     month  idr_eur_rate
0  2014-11  15198.200000
1  2014-12  15316.347826
2  2015-01  14637.318182


### 2.3 Inflasi Bulanan Bali 2009–2025

In [18]:
inflasi_raw = pd.read_excel('../data/raw/Data Inflasi.xlsx', skiprows=4)
inflasi_raw.columns = ['no', 'periode', 'inflasi', 'extra']
inflasi_raw = inflasi_raw[['no', 'periode', 'inflasi']].copy()

print('Shape raw:', inflasi_raw.shape)
print(inflasi_raw.head(5))
print(inflasi_raw.tail(5))


Shape raw: (204, 3)
   no         periode inflasi
0   1   Desember 2025  2.92 %
1   2   November 2025  2.72 %
2   3    Oktober 2025  2.86 %
3   4  September 2025  2.65 %
4   5    Agustus 2025  2.31 %
      no        periode inflasi
199  200       Mei 2009  6.04 %
200  201     April 2009  7.31 %
201  202     Maret 2009  7.92 %
202  203  Februari 2009   8.6 %
203  204   Januari 2009  9.17 %


In [19]:
bulan_map_inflasi = {
    'Januari': '01', 'Februari': '02', 'Maret': '03',
    'April': '04', 'Mei': '05', 'Juni': '06', 'Juli': '07',
    'Agustus': '08', 'September': '09', 'Oktober': '10',
    'November': '11', 'Desember': '12'
}

def parse_periode(periode_str):
    """Ubah 'Desember 2025' -> '2025-12-01'"""
    parts = str(periode_str).strip().split()
    if len(parts) == 2:
        bulan_str, tahun_str = parts[0], parts[1]
        bulan_num = bulan_map_inflasi.get(bulan_str)
        if bulan_num and tahun_str.isdigit():
            return f"{tahun_str}-{bulan_num}-01"
    return None

inflasi_df = inflasi_raw[inflasi_raw['periode'].notna()].copy()
inflasi_df = inflasi_df[inflasi_df['no'].apply(lambda x: str(x).isdigit() or isinstance(x, (int, float)))].copy()

inflasi_df['inflasi_processed'] = (
    inflasi_df['inflasi']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors='coerce')
)

inflasi_df['date'] = inflasi_df['periode'].apply(parse_periode)
inflasi_df = inflasi_df[inflasi_df['date'].notna()].copy()
inflasi_df['date'] = pd.to_datetime(inflasi_df['date'])
inflasi_df['month'] = inflasi_df['date'].dt.to_period('M')
inflasi_df = inflasi_df[['date', 'month', 'inflasi_processed']].copy()

# Sort ascending & filter rentang 2009-2025
inflasi_df = inflasi_df.sort_values('date').reset_index(drop=True)
inflasi_df = inflasi_df[
    (inflasi_df['date'].dt.year >= 2009) &
    (inflasi_df['date'].dt.year <= 2025)
].reset_index(drop=True)

print('Shape final  :', inflasi_df.shape)
print('Rentang data :', inflasi_df['date'].min(), '->', inflasi_df['date'].max())
print(f'Jumlah bulan : {inflasi_df.shape[0]} (expected: 204)')
inflasi_df.head()


Shape final  : (204, 3)
Rentang data : 2009-01-01 00:00:00 -> 2025-12-01 00:00:00
Jumlah bulan : 204 (expected: 204)


,date,month,inflasi_processed
0,2009-01-01,2009-01,9.17
1,2009-02-01,2009-02,8.60
2,2009-03-01,2009-03,7.92
3,2009-04-01,2009-04,7.31
4,2009-05-01,2009-05,6.04


In [20]:
# NB asli (02_1) menyimpan inflasi_df dua kali dengan kode identik (cell 28 & 31).
# Di sini disederhanakan jadi satu kali save.
inflasi_df.to_csv('../data/processed/inflasi_clean.csv', index=False)
print('✓ inflasi_clean.csv disimpan')
print(f'   Shape  : {inflasi_df.shape}')
print(f'   Rentang: {inflasi_df["month"].min()} -> {inflasi_df["month"].max()}')
print(f'   Missing: {inflasi_df.isnull().sum().sum()}')


✓ inflasi_clean.csv disimpan
   Shape  : (204, 3)
   Rentang: 2009-01 -> 2025-12
   Missing: 0


### 2.4 World Bank — Indeks Ekonomi Global Tertimbang
Dipakai sebagai proxy kondisi ekonomi negara asal wisatawan utama Bali (`economic_risk_score`).

In [21]:
wb = pd.read_csv("../data/raw/worldbank_2026-06-10.csv")
wb["date"] = pd.to_datetime(wb["date"])

# Handle 2025: composite_gdp_weighted = 0 -> NaN
wb.loc[wb["composite_gdp_weighted"] == 0, "composite_gdp_weighted"] = np.nan

wb_pivot = wb.pivot_table(index="date", columns="country", values="composite_gdp_weighted")

# Interpolasi linear untuk 2025, fallback ffill
wb_pivot = wb_pivot.interpolate(method="linear", limit_direction="forward")
wb_pivot = wb_pivot.ffill()

wb_pivot.head()


country,AUS,CHN,DEU,GBR,IND,JPN,MYS,SGP,USA
date,,,,,,,,,
2009-01-01,1.469468,1.469468,1.469468,1.469468,1.469468,1.469468,1.469468,1.469468,1.469468
2010-01-01,5.627735,5.627735,5.627735,5.627735,5.627735,5.627735,5.627735,5.627735,5.627735
2011-01-01,4.072502,4.072502,4.072502,4.072502,4.072502,4.072502,4.072502,4.072502,4.072502
2012-01-01,4.084882,4.084882,4.084882,4.084882,4.084882,4.084882,4.084882,4.084882,4.084882
2013-01-01,3.861158,3.861158,3.861158,3.861158,3.861158,3.861158,3.861158,3.861158,3.861158


In [22]:
# Bobot kasar berdasarkan estimasi porsi wisman per negara asal
TOURIST_WEIGHT = {
    "AUS": 0.20, "CHN": 0.18, "GBR": 0.08, "USA": 0.08,
    "DEU": 0.07, "JPN": 0.10, "IND": 0.12, "MYS": 0.10, "SGP": 0.07,
}

available_countries = [c for c in TOURIST_WEIGHT if c in wb_pivot.columns]
weights = np.array([TOURIST_WEIGHT[c] for c in available_countries])
weights = weights / weights.sum()

wb_pivot["economic_index"] = wb_pivot[available_countries].values @ weights

# Normalisasi: GDP rendah = risiko tinggi -> dibalik
wb_pivot["economic_risk_score"] = 1 - (
    (wb_pivot["economic_index"] - wb_pivot["economic_index"].min()) /
    (wb_pivot["economic_index"].max() - wb_pivot["economic_index"].min())
)

# Expand tahunan -> bulanan
wb_monthly = wb_pivot[["economic_index", "economic_risk_score"]].resample("MS").ffill()
wb_monthly.head(14)


country,economic_index,economic_risk_score
date,,
2009-01-01,1.469468,0.520559
2009-02-01,1.469468,0.520559
2009-03-01,1.469468,0.520559
2009-04-01,1.469468,0.520559
2009-05-01,1.469468,0.520559
2009-06-01,1.469468,0.520559
2009-07-01,1.469468,0.520559
2009-08-01,1.469468,0.520559
2009-09-01,1.469468,0.520559


In [23]:
# ── PATCH: Extend wb_monthly sampai Des 2025 (ffill struktural) ───────────────
# World Bank merilis data tahunan dengan lag. Nilai terakhir di-ffill ke depan
# sebagai asumsi stabil (economic_risk_score sebagai indikator struktural).
# `wb_is_imputed` DIPERTAHANKAN (tidak di-drop) sebagai flag transparansi —
# supaya jelas bulan mana yang nilainya hasil imputasi, bukan data asli World Bank.
_wb_last_real  = wb_monthly.index.max()
_wb_extend_end = pd.Timestamp('2025-12-01')

if _wb_last_real < _wb_extend_end:
    _ext_idx = pd.date_range(
        start = _wb_last_real + pd.DateOffset(months=1),
        end   = _wb_extend_end,
        freq  = 'MS'
    )
    _ext_df = pd.DataFrame(index=_ext_idx, columns=wb_monthly.columns)
    wb_monthly = pd.concat([wb_monthly, _ext_df]).ffill()
    wb_monthly['wb_is_imputed'] = (wb_monthly.index > _wb_last_real).astype(int)
    print(f"wb_monthly extended: {_wb_last_real.date()} -> {wb_monthly.index.max().date()}")
    print(f"Imputed months     : {wb_monthly['wb_is_imputed'].sum()}")
else:
    wb_monthly['wb_is_imputed'] = 0
    print(f"wb_monthly sudah cukup: {_wb_last_real.date()}")

wb_monthly.tail(5)


wb_monthly extended: 2024-01-01 -> 2025-12-01
Imputed months     : 23


country,economic_index,economic_risk_score,wb_is_imputed
2025-08-01,2.779713,0.360437,1
2025-09-01,2.779713,0.360437,1
2025-10-01,2.779713,0.360437,1
2025-11-01,2.779713,0.360437,1
2025-12-01,2.779713,0.360437,1


In [24]:
wb_monthly.to_csv('../data/processed/wb_monthly_economic.csv')
print('✓ wb_monthly_economic.csv disimpan (termasuk kolom wb_is_imputed untuk transparansi)')


✓ wb_monthly_economic.csv disimpan (termasuk kolom wb_is_imputed untuk transparansi)


---
## 3. Tema: Akomodasi

Tingkat Penghunian Kamar (TPK) Hotel Bintang & Non Bintang — indikator langsung tingkat okupansi
pariwisata di Bali, bagian dari `crisis_component_tourism`.


In [25]:
excel_bintang     = '../data/raw/Tingkat Penghunian Kamar (TPK) Hotel Bintang.xlsx'
excel_non_bintang = '../data/raw/Tingkat Penghunian Kamar (TPK).xlsx'

xl_b  = pd.ExcelFile(excel_bintang)
xl_nb = pd.ExcelFile(excel_non_bintang)

print(f'Sheet Hotel Bintang: {xl_b.sheet_names}')
print(f'Sheet Non Bintang  : {xl_nb.sheet_names}')

kelas_valid = ['Bintang 5', 'Bintang 4', 'Bintang 3', 'Bintang 2', 'Bintang 1', 'Jumlah / Total']
kabupaten_valid = [
    'Kab. Jembrana', 'Kab. Tabanan', 'Kab. Badung', 'Kab. Gianyar',
    'Kab. Klungkung', 'Kab. Bangli', 'Kab. Karangasem', 'Kab. Buleleng',
    'Kota Denpasar', 'Provinsi Bali'
]


Sheet Hotel Bintang: ['2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
Sheet Non Bintang  : ['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019']


In [26]:
bulan_map_tpk = {
    'Januari': '01', 'Pebruari': '02', 'Februari': '02', 'Maret': '03',
    'April': '04', 'Mei': '05', 'Juni': '06', 'Juli': '07',
    'Agustus': '08', 'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
}

# --- Proses Hotel Bintang ---
records_bintang = []
for sheet in xl_b.sheet_names:
    raw = pd.read_excel(excel_bintang, sheet_name=sheet, header=None)
    tahun_cell = str(raw.iloc[2, 1]).strip()
    bulan_headers = raw.iloc[3, 1:13].tolist()

    data = raw.iloc[4:].copy()
    data.columns = ['kelas'] + bulan_headers + ['tahunan'] + list(range(len(data.columns)-14))
    data = data[data['kelas'].astype(str).str.strip().isin(kelas_valid)].copy()

    for bulan in bulan_headers:
        bulan_str = str(bulan).strip()
        if bulan_str in bulan_map_tpk:
            values = pd.to_numeric(data[bulan], errors='coerce').dropna()
            if len(values) > 0:
                records_bintang.append({
                    'month_str': f"{tahun_cell}-{bulan_map_tpk[bulan_str]}-01",
                    'tpk_bintang': values.mean()
                })
    print(f'✓ [Bintang] Sheet {sheet} diproses')

# --- Proses Non Bintang ---
records_non_bintang = []
for sheet in xl_nb.sheet_names:
    raw = pd.read_excel(excel_non_bintang, sheet_name=sheet, header=None)
    tahun_cell = str(raw.iloc[2, 1]).strip()
    bulan_headers = raw.iloc[3, 1:13].tolist()

    data = raw.iloc[4:].copy()
    data.columns = ['kabupaten'] + bulan_headers + ['tahunan'] + list(range(len(data.columns)-14))
    data = data[data['kabupaten'].astype(str).str.strip().isin(kabupaten_valid)].copy()

    for bulan in bulan_headers:
        bulan_str = str(bulan).strip()
        if bulan_str in bulan_map_tpk:
            row_provinsi = data[data['kabupaten'].astype(str).str.strip() == 'Provinsi Bali']
            value = pd.to_numeric(row_provinsi[bulan].values[0], errors='coerce') if len(row_provinsi) > 0 else np.nan
            records_non_bintang.append({
                'month_str': f"{tahun_cell}-{bulan_map_tpk[bulan_str]}-01",
                'tpk_non_bintang': value
            })
    print(f'✓ [Non Bintang] Sheet {sheet} diproses')

# --- Merge ---
df_bintang     = pd.DataFrame(records_bintang)
df_non_bintang = pd.DataFrame(records_non_bintang)

tpk_df = pd.merge(df_bintang, df_non_bintang, on='month_str', how='outer')
tpk_df['date']  = pd.to_datetime(tpk_df['month_str'])
tpk_df = tpk_df.sort_values('date').reset_index(drop=True)
tpk_df['month'] = tpk_df['date'].dt.to_period('M')
tpk_df = tpk_df[['date', 'month', 'tpk_bintang', 'tpk_non_bintang']]

print(f'\nRentang data       : {tpk_df["date"].min()} sampai {tpk_df["date"].max()}')
print(f'Total baris        : {len(tpk_df)}')
print(f'Missing bintang    : {tpk_df["tpk_bintang"].isna().sum()}')
print(f'Missing non-bintang: {tpk_df["tpk_non_bintang"].isna().sum()}')


✓ [Bintang] Sheet 2000 diproses
✓ [Bintang] Sheet 2001 diproses
✓ [Bintang] Sheet 2002 diproses
✓ [Bintang] Sheet 2003 diproses
✓ [Bintang] Sheet 2004 diproses
✓ [Bintang] Sheet 2005 diproses
✓ [Bintang] Sheet 2006 diproses
✓ [Bintang] Sheet 2007 diproses
✓ [Bintang] Sheet 2008 diproses
✓ [Bintang] Sheet 2009 diproses
✓ [Bintang] Sheet 2010 diproses
✓ [Bintang] Sheet 2011 diproses
✓ [Bintang] Sheet 2012 diproses
✓ [Bintang] Sheet 2013 diproses
✓ [Bintang] Sheet 2014 diproses
✓ [Bintang] Sheet 2015 diproses
✓ [Bintang] Sheet 2016 diproses
✓ [Bintang] Sheet 2017 diproses
✓ [Bintang] Sheet 2018 diproses
✓ [Bintang] Sheet 2019 diproses
✓ [Bintang] Sheet 2020 diproses
✓ [Bintang] Sheet 2021 diproses
✓ [Bintang] Sheet 2022 diproses
✓ [Bintang] Sheet 2023 diproses
✓ [Bintang] Sheet 2024 diproses
✓ [Bintang] Sheet 2025 diproses
✓ [Bintang] Sheet 2026 diproses
✓ [Non Bintang] Sheet 2007 diproses
✓ [Non Bintang] Sheet 2008 diproses
✓ [Non Bintang] Sheet 2009 diproses
✓ [Non Bintang] Sheet 2010 d

In [27]:
tpk_df.to_csv('../data/processed/tpk_clean.csv', index=False)
print('✓ tpk_clean.csv disimpan')


✓ tpk_clean.csv disimpan


In [28]:
# ── Validation: TPK Hotel ─────────────────────────────────
print('=== VALIDATION: TPK Hotel ===')
print(f'Shape         : {tpk_df.shape}')
print(f'Rentang       : {tpk_df["month"].min()} -> {tpk_df["month"].max()}')
print(f'Missing       :\n{tpk_df.isnull().sum().to_string()}')
print(f'Duplikat month: {tpk_df["month"].duplicated().sum()}')
print()
print('Preview:')
print(tpk_df.head(3).to_string())


=== VALIDATION: TPK Hotel ===
Shape         : (315, 4)
Rentang       : 2000-01 -> 2026-03
Missing       :
date                 0
month                0
tpk_bintang          0
tpk_non_bintang    160
Duplikat month: 0

Preview:
        date    month  tpk_bintang  tpk_non_bintang
0 2000-01-01  2000-01       47.144              NaN
1 2000-02-01  2000-02       47.366              NaN
2 2000-03-01  2000-03       44.950              NaN


### 3.2 Lama Menginap Hotel Bintang & Non Bintang

Rata-rata lama menginap (hari) adalah sinyal kualitas kunjungan — berbeda dari TPK (okupansi).
Lama menginap yang **memendek** menunjukkan penurunan engagement wisatawan bahkan sebelum
angka kunjungan (wisman) turun, sehingga berguna sebagai leading indicator dalam `crisis_component_tourism`.

Format file identik dengan TPK: wide table BPS, baris = kelas hotel / kabupaten, kolom = bulan.
Parsing menggunakan logika yang sama dengan Section 3.1.

**Catatan data:**
- `lama_bintang`: per kelas hotel (Bintang 1–5 + Total), mulai tahun 2000 → di-aggregate rata-rata semua kelas per bulan.
- `lama_nonbintang`: per kabupaten/kota Bali, mulai tahun 2007 → di-aggregate rata-rata semua kabupaten per bulan (kecuali `Provinsi Bali` jika tersedia sebagai aggregate resmi).


In [29]:
excel_lama_bintang     = '../data/raw/Rata-Rata Lama Menginap Tamu Asing dan Domestik pada Hotel Bintang.xlsx'
excel_lama_nonbintang  = '../data/raw/Rata-Rata Lama Menginap Tamu Asing dan Domestik pada Hotel Non Bintang.xlsx'

xl_lb  = pd.ExcelFile(excel_lama_bintang)
xl_lnb = pd.ExcelFile(excel_lama_nonbintang)

print(f'Sheet Lama Menginap Bintang    : {xl_lb.sheet_names}')
print(f'Sheet Lama Menginap Non Bintang: {xl_lnb.sheet_names}')

# Bintang: kelas valid
kelas_valid_lama = ['Bintang 5', 'Bintang 4', 'Bintang 3', 'Bintang 2', 'Bintang 1']

# Non-bintang: semua kabupaten (kecuali 'Provinsi Bali' — jika ada, ambil itu saja)
kabupaten_valid_lama = [
    'Kab. Jembrana', 'Kab. Tabanan', 'Kab. Badung', 'Kab. Gianyar',
    'Kab. Klungkung', 'Kab. Bangli', 'Kab. Karangasem', 'Kab. Buleleng',
    'Kota Denpasar'
]


Sheet Lama Menginap Bintang    : ['2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
Sheet Lama Menginap Non Bintang: ['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019']


In [30]:
# --- Proses Lama Menginap Hotel Bintang ---
records_lama_bintang = []
for sheet in xl_lb.sheet_names:
    raw = pd.read_excel(excel_lama_bintang, sheet_name=sheet, header=None)
    tahun_cell   = str(raw.iloc[2, 1]).strip()
    bulan_headers = raw.iloc[3, 1:13].tolist()

    data = raw.iloc[4:].copy()
    data.columns = ['kelas'] + bulan_headers + ['tahunan'] + list(range(len(data.columns) - 14))
    data = data[data['kelas'].astype(str).str.strip().isin(kelas_valid_lama)].copy()

    for bulan in bulan_headers:
        bulan_str = str(bulan).strip()
        if bulan_str in bulan_map_tpk:
            values = pd.to_numeric(data[bulan], errors='coerce').dropna()
            if len(values) > 0:
                records_lama_bintang.append({
                    'month_str': f"{tahun_cell}-{bulan_map_tpk[bulan_str]}-01",
                    'lama_menginap_bintang': values.mean()
                })
    print(f'✓ [Lama Bintang] Sheet {sheet} diproses')

# --- Proses Lama Menginap Hotel Non Bintang ---
records_lama_nonbintang = []
for sheet in xl_lnb.sheet_names:
    raw = pd.read_excel(excel_lama_nonbintang, sheet_name=sheet, header=None)
    tahun_cell   = str(raw.iloc[2, 1]).strip()
    bulan_headers = raw.iloc[3, 1:13].tolist()

    data = raw.iloc[4:].copy()
    data.columns = ['kabupaten'] + bulan_headers + ['tahunan'] + list(range(len(data.columns) - 14))

    # Cek apakah ada row 'Provinsi Bali' sebagai aggregate resmi
    row_prov = data[data['kabupaten'].astype(str).str.strip() == 'Provinsi Bali']

    for bulan in bulan_headers:
        bulan_str = str(bulan).strip()
        if bulan_str not in bulan_map_tpk:
            continue
        if len(row_prov) > 0:
            # Pakai nilai Provinsi Bali kalau ada
            value = pd.to_numeric(row_prov[bulan].values[0], errors='coerce')
        else:
            # Fallback: rata-rata semua kabupaten
            subset = data[data['kabupaten'].astype(str).str.strip().isin(kabupaten_valid_lama)]
            values = pd.to_numeric(subset[bulan], errors='coerce').dropna()
            value  = values.mean() if len(values) > 0 else np.nan
        records_lama_nonbintang.append({
            'month_str': f"{tahun_cell}-{bulan_map_tpk[bulan_str]}-01",
            'lama_menginap_non_bintang': value
        })
    print(f'✓ [Lama Non Bintang] Sheet {sheet} diproses')

# --- Gabung & Bersihkan ---
df_lama_b   = pd.DataFrame(records_lama_bintang)
df_lama_nb  = pd.DataFrame(records_lama_nonbintang)

lama_df = pd.merge(df_lama_b, df_lama_nb, on='month_str', how='outer')
lama_df['date']  = pd.to_datetime(lama_df['month_str'])
lama_df = lama_df.sort_values('date').reset_index(drop=True)
lama_df['month'] = lama_df['date'].dt.to_period('M')

# Filter rentang 2009-2025 agar konsisten dengan dataset inti
lama_df = lama_df[
    (lama_df['date'].dt.year >= 2009) &
    (lama_df['date'].dt.year <= 2025)
].reset_index(drop=True)

lama_df = lama_df[['date', 'month', 'lama_menginap_bintang', 'lama_menginap_non_bintang']]

print(f'Rentang data          : {lama_df["date"].min()} sampai {lama_df["date"].max()}')
print(f'Total baris           : {len(lama_df)}')
print(f'Missing bintang       : {lama_df["lama_menginap_bintang"].isna().sum()}')
print(f'Missing non-bintang   : {lama_df["lama_menginap_non_bintang"].isna().sum()}')
lama_df.head()


✓ [Lama Bintang] Sheet 2000 diproses
✓ [Lama Bintang] Sheet 2001 diproses
✓ [Lama Bintang] Sheet 2002 diproses
✓ [Lama Bintang] Sheet 2003 diproses
✓ [Lama Bintang] Sheet 2004 diproses
✓ [Lama Bintang] Sheet 2005 diproses
✓ [Lama Bintang] Sheet 2006 diproses
✓ [Lama Bintang] Sheet 2007 diproses
✓ [Lama Bintang] Sheet 2008 diproses
✓ [Lama Bintang] Sheet 2009 diproses
✓ [Lama Bintang] Sheet 2010 diproses
✓ [Lama Bintang] Sheet 2011 diproses
✓ [Lama Bintang] Sheet 2012 diproses
✓ [Lama Bintang] Sheet 2013 diproses
✓ [Lama Bintang] Sheet 2014 diproses
✓ [Lama Bintang] Sheet 2015 diproses
✓ [Lama Bintang] Sheet 2016 diproses
✓ [Lama Bintang] Sheet 2017 diproses
✓ [Lama Bintang] Sheet 2018 diproses
✓ [Lama Bintang] Sheet 2019 diproses
✓ [Lama Bintang] Sheet 2020 diproses
✓ [Lama Bintang] Sheet 2021 diproses
✓ [Lama Bintang] Sheet 2022 diproses
✓ [Lama Bintang] Sheet 2023 diproses
✓ [Lama Bintang] Sheet 2024 diproses
✓ [Lama Bintang] Sheet 2025 diproses
✓ [Lama Bintang] Sheet 2026 diproses
✓

,date,month,lama_menginap_bintang,lama_menginap_non_bintang
0,2009-01-01,2009-01,3.082,2.82
1,2009-02-01,2009-02,3.422,2.55
2,2009-03-01,2009-03,3.792,3.17
3,2009-04-01,2009-04,3.614,2.92
4,2009-05-01,2009-05,3.900,2.42


In [31]:
lama_df.to_csv('../data/processed/lama_menginap_clean.csv', index=False)
print('✓ lama_menginap_clean.csv disimpan')
print(f'   Shape  : {lama_df.shape}')
print(f'   Rentang: {lama_df["month"].min()} -> {lama_df["month"].max()}')
print(f'   Missing: {lama_df.isnull().sum().sum()}')


✓ lama_menginap_clean.csv disimpan
   Shape  : (204, 4)
   Rentang: 2009-01 -> 2025-12
   Missing: 72


In [32]:
# ── Validation: Lama Menginap ─────────────────────────────────────────────────
print('=== VALIDATION: Lama Menginap Hotel ===')
print(f'Shape         : {lama_df.shape}')
print(f'Rentang       : {lama_df["month"].min()} -> {lama_df["month"].max()}')
print(f'Missing       :\n{lama_df.isnull().sum().to_string()}')
print(f'Duplikat month: {lama_df["month"].duplicated().sum()}')
print()
print('Stats lama menginap (hari):')
print(lama_df[['lama_menginap_bintang', 'lama_menginap_non_bintang']].describe().round(2).to_string())
print()
print('Preview:')
print(lama_df.head(3).to_string())


=== VALIDATION: Lama Menginap Hotel ===
Shape         : (204, 4)
Rentang       : 2009-01 -> 2025-12
Missing       :
date                          0
month                         0
lama_menginap_bintang         0
lama_menginap_non_bintang    72
Duplikat month: 0

Stats lama menginap (hari):
       lama_menginap_bintang  lama_menginap_non_bintang
count                 204.00                     132.00
mean                    2.89                       2.63
std                     0.53                       0.25
min                     1.68                       2.01
25%                     2.56                       2.48
50%                     2.94                       2.62
75%                     3.22                       2.76
max                     4.08                       3.47

Preview:
        date    month  lama_menginap_bintang  lama_menginap_non_bintang
0 2009-01-01  2009-01                  3.082                       2.82
1 2009-02-01  2009-02                  3.422       

---
## 4. Tema: Risiko Eksternal / Bencana

Sinyal gempa bumi (USGS), cuaca ekstrem (Open-Meteo), dan fitur historis gabungan BaliGuard
(gempa + gunung berapi + cuaca). Komponen ini menyusun `physical_risk_score`.


### 4.1 Earthquake (USGS)

In [33]:
eq_raw = pd.read_csv("../data/raw/weather/usgs_bali_earthquakes_2009_present.csv")
eq = eq_raw.copy()
eq["date"] = pd.to_datetime(eq["date"])

# Drop kolom redundant/konstan: "source" (konstan), "year" (derivable dari date)
eq = eq.drop(columns=["source", "year"])

# Hitung ulang distance_to_bali_km via Haversine — kolom original berselisih
# ~9 km dari kalkulasi ini (kemungkinan reference point sumber berbeda).
BALI_LAT, BALI_LON = -8.3405, 115.0920

def haversine_km(lat, lon, ref_lat=BALI_LAT, ref_lon=BALI_LON):
    R = 6371.0
    dlat = np.radians(ref_lat - lat)
    dlon = np.radians(ref_lon - lon)
    a = (np.sin(dlat/2)**2
         + np.cos(np.radians(lat)) * np.cos(np.radians(ref_lat)) * np.sin(dlon/2)**2)
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

eq["distance_to_bali_km"] = haversine_km(eq["latitude"].values, eq["longitude"].values)

# Flag gempa sangat dangkal (< 5 km) — diflag untuk traceability, tidak di-drop
eq["depth_shallow_flag"] = (eq["depth_km"] < 5).astype(int)

# Proxy energi seismik (Gutenberg-Richter): 10^(1.5*M)
eq["seismic_energy_proxy"] = 10 ** (1.5 * eq["magnitude"])

# ── Agregasi harian -> bulanan ────────────────────────────────────────────────
eq_daily = (
    eq.groupby("date")
    .agg(
        eq_count=("event_id", "count"),
        eq_max_magnitude=("magnitude", "max"),
        eq_mean_depth=("depth_km", "mean"),
        eq_seismic_energy=("seismic_energy_proxy", "sum"),
        eq_shallow_flag=("depth_shallow_flag", "max"),
    )
)

eq_monthly = eq_daily.resample("MS").agg(
    eq_count=("eq_count", "sum"),
    eq_max_magnitude=("eq_max_magnitude", "max"),
    eq_mean_depth=("eq_mean_depth", "mean"),
    eq_seismic_energy=("eq_seismic_energy", "sum"),
    eq_shallow_flag=("eq_shallow_flag", "max"),
)

# ── Composite seismic risk score: energy (70%) + frekuensi (30%) ─────────────
eq_monthly["eq_energy_norm"] = (
    (eq_monthly["eq_seismic_energy"] - eq_monthly["eq_seismic_energy"].min()) /
    (eq_monthly["eq_seismic_energy"].max() - eq_monthly["eq_seismic_energy"].min())
)
eq_monthly["eq_count_norm"] = (
    (eq_monthly["eq_count"] - eq_monthly["eq_count"].min()) /
    (eq_monthly["eq_count"].max() - eq_monthly["eq_count"].min())
)
eq_monthly["eq_risk_score"] = (
    0.7 * eq_monthly["eq_energy_norm"] + 0.3 * eq_monthly["eq_count_norm"]
)

eq_monthly[["eq_count", "eq_max_magnitude", "eq_seismic_energy", "eq_risk_score"]].head()


,eq_count,eq_max_magnitude,eq_seismic_energy,eq_risk_score
date,,,,
2009-01-01,6,5.1,1.638291e+08,0.013508
2009-02-01,1,4.0,1.000000e+06,0.000000
2009-03-01,1,4.9,2.238721e+07,0.000293
2009-04-01,2,4.8,1.866731e+07,0.002498
2009-05-01,1,4.6,7.943282e+06,0.000095


### 4.2 Weather (Open-Meteo)

In [34]:
wx_raw = pd.read_csv("../data/raw/weather/open_meteo_bali_weather_extreme_2009_present.csv")
wx = wx_raw.copy()
wx["date"] = pd.to_datetime(wx["time"])

# Drop kolom redundant/konstan (verified saat EDA):
# rain_sum identik precipitation_sum; lat/lon/timezone/source konstan; time -> date
wx = wx.drop(columns=["time", "rain_sum", "latitude", "longitude", "timezone", "source"])

# Log1p untuk distribusi zero-inflated (precip & extreme score)
wx["extreme_score_log"] = np.log1p(wx["extreme_weather_score"])
wx["precip_log"]        = np.log1p(wx["precipitation_sum"])

wx = wx.set_index("date").sort_index()
wx_monthly = (
    wx.resample("MS").agg(
        wx_precip_sum=("precipitation_sum", "sum"),
        wx_humidity_mean=("relative_humidity_2m_mean", "mean"),
    )
)
wx_monthly.head()


,wx_precip_sum,wx_humidity_mean
date,,
2009-01-01,361.1,90.709677
2009-02-01,375.6,91.785714
2009-03-01,223.2,89.000000
2009-04-01,114.1,88.466667
2009-05-01,200.4,90.451613


### 4.3 BaliGuard Historical Monthly Features (Gempa + Gunung Berapi + Cuaca)

In [35]:
bg_raw = pd.read_csv("../data/raw/weather/baliguard_historical_monthly_features_2009_present.csv")
bg = bg_raw.copy()
bg["month"] = pd.to_datetime(bg["month"])
bg = bg.set_index("month").sort_index()

# weather_max_precipitation identik 100% dengan weather_max_rain (verified via EDA)
bg = bg.drop(columns=["weather_max_precipitation"])

# Catatan: pvmbg_volcano_status == max(agung, batur) — derived column.
# JANGAN dijumlahkan dengan volcano_status_agung & batur (triple-counting).

# Seismic energy proxy dari BMKG max magnitude (Gutenberg-Richter)
bg["bmkg_seismic_energy"] = 10 ** (1.5 * bg["bmkg_max_magnitude_30d"])

# Log transform kolom zero-inflated / heavy right-skew
bg["eq_count_log"]  = np.log1p(bg["earthquake_count"])
bg["eq_m5plus_log"] = np.log1p(bg["earthquake_count_m5_plus"])
bg["rain_log"]      = np.log1p(bg["weather_max_rain"])

def minmax_norm(s):
    denom = s.max() - s.min()
    return (s - s.min()) / denom if denom != 0 else pd.Series(0, index=s.index)

# Composite disaster risk score: 40% seismik + 30% gunung berapi + 30% cuaca
bg["seismic_norm"] = minmax_norm(bg["bmkg_seismic_energy"])
bg["volcano_norm"] = minmax_norm(bg["pvmbg_volcano_status"])
bg["weather_severity_norm"] = (
    0.4 * minmax_norm(bg["bmkg_extreme_weather_days"]) +
    0.4 * minmax_norm(bg["weather_max_rain"]) +
    0.2 * minmax_norm(bg["weather_max_wind_gust"])
)
bg["disaster_risk_score"] = (
    0.4 * bg["seismic_norm"] + 0.3 * bg["volcano_norm"] + 0.3 * bg["weather_severity_norm"]
)

bg[["bmkg_max_magnitude_30d", "pvmbg_volcano_status",
    "bmkg_extreme_weather_days", "disaster_risk_score"]].head(10)


,bmkg_max_magnitude_30d,pvmbg_volcano_status,bmkg_extreme_weather_days,disaster_risk_score
month,,,,
2009-01-01,5.1,0,6,0.089349
2009-02-01,4.0,0,4,0.092702
2009-03-01,4.9,0,2,0.037040
2009-04-01,4.8,0,1,0.026324
2009-05-01,4.6,0,1,0.033700
2009-06-01,5.5,0,0,0.021086
2009-07-01,5.9,0,0,0.035900
2009-08-01,4.5,0,0,0.030300
2009-09-01,5.7,0,1,0.046395


---
## 5. Tema: Sinyal Digital & Media

Sentimen media global (GDELT) dan minat pencarian publik (Google Trends). Komponen ini menyusun
`media_risk_score` dan `tourist_perception_score`.


### 5.1 GDELT — Sentimen & Volume Pemberitaan

In [36]:
gdelt_hist   = pd.read_csv("../data/raw/gdelt/gdelt_historical_2009_2021.csv")
gdelt_recent = pd.read_csv("../data/raw/gdelt/gdelt_2026_W24.csv")

gdelt = (
    pd.concat([gdelt_hist, gdelt_recent])
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .drop_duplicates(subset=["date"], keep="last")  # recent menang kalau overlap
    .set_index("date")
    .sort_index()
)

# File tambahan hanya menyumbang 1 kolom baru masing-masing (kolom lain identik)
gdelt_crime = (
    pd.read_csv("../data/raw/gdelt/gdelt_historical_2009_2025.csv")
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .set_index("date")[["crime_count"]]
)
gdelt_env = (
    pd.read_csv("../data/raw/gdelt/gdelt_historical_env_2009_2025.csv")
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .set_index("date")[["issue_article_count"]]
)

gdelt = gdelt.join(gdelt_crime, how="left").join(gdelt_env, how="left")

gdelt["crime_ratio"] = gdelt["crime_count"] / gdelt["article_count"].replace(0, np.nan)
gdelt["issue_ratio"] = gdelt["issue_article_count"] / gdelt["article_count"].replace(0, np.nan)
gdelt["risk_ratio"]  = gdelt["risk_article_count"] / gdelt["article_count"].replace(0, np.nan)

# Normalisasi tone: makin negatif -> makin tinggi risiko -> dibalik
gdelt["tone_risk_score"] = (gdelt["avg_tone"] - gdelt["avg_tone"].max()) / \
                           (gdelt["avg_tone"].min() - gdelt["avg_tone"].max())
gdelt["risk_ratio_score"] = (gdelt["risk_ratio"] - gdelt["risk_ratio"].min()) / \
                             (gdelt["risk_ratio"].max() - gdelt["risk_ratio"].min())

gdelt["gdelt_crisis_score"] = (gdelt["tone_risk_score"] + gdelt["risk_ratio_score"]) / 2

gdelt[["avg_tone", "risk_ratio", "tone_risk_score", "risk_ratio_score", "gdelt_crisis_score"]].head()


,avg_tone,risk_ratio,tone_risk_score,risk_ratio_score,gdelt_crisis_score
date,,,,,
2009-01-01,4.881555,0.003690,0.120732,0.030061,0.075397
2009-02-01,4.760496,0.007712,0.131566,0.062827,0.097197
2009-03-01,5.302634,0.068852,0.083048,0.560913,0.321980
2009-04-01,4.192748,0.025490,0.182377,0.207658,0.195018
2009-05-01,5.349026,0.000000,0.078896,0.000000,0.039448


### 5.2 Google Trends — Minat Pencarian Publik

In [37]:
gtrends_hist     = pd.read_csv("../data/raw/gtrends/gtrends_historical_2009_2021.csv")
gtrends_recent   = pd.read_csv("../data/raw/gtrends/gtrends_2026_W24.csv")
gtrends_combined = pd.read_csv("../data/raw/gtrends/gtrends_positive_combined_historical_2009_2025_2.csv")

gtrends = (
    pd.concat([gtrends_hist, gtrends_recent, gtrends_combined])
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .drop_duplicates(subset=["date"], keep="last")
    .set_index("date")
    .sort_index()
)

gtrends_monthly = gtrends.resample("MS").mean()

keyword_cols = list(gtrends_monthly.columns)
print(f"Total keyword columns: {len(keyword_cols)}")
print(keyword_cols)

gtrends_monthly["kw_coverage"] = gtrends_monthly[keyword_cols].notna().sum(axis=1)
gtrends_monthly["kw_coverage_pct"] = gtrends_monthly["kw_coverage"] / len(keyword_cols)
gtrends_monthly["low_coverage_flag"] = (gtrends_monthly["kw_coverage_pct"] < 0.5).astype(int)

print("\nCoverage per periode:")
print(gtrends_monthly[["kw_coverage", "kw_coverage_pct", "low_coverage_flag"]].describe().round(2))

# Composite: rata-rata keyword yang tersedia (NaN otomatis di-skip)
gtrends_monthly["trend_composite"] = gtrends_monthly[keyword_cols].mean(axis=1)

_min = gtrends_monthly["trend_composite"].min()
_max = gtrends_monthly["trend_composite"].max()
gtrends_monthly["trend_risk_score"] = 1 - ((gtrends_monthly["trend_composite"] - _min) / (_max - _min))

gtrends_monthly["trend_rolling_mean"] = gtrends_monthly["trend_composite"].rolling(3, min_periods=2).mean()
gtrends_monthly["trend_drop_flag"] = (
    gtrends_monthly["trend_composite"] < gtrends_monthly["trend_rolling_mean"] * 0.75
).astype(int)

output_cols = [
    "trend_composite", "trend_risk_score", "trend_rolling_mean", "trend_drop_flag",
    "kw_coverage", "kw_coverage_pct", "low_coverage_flag"
]
gtrends_monthly[output_cols].head(10)


Total keyword columns: 26
['Bali vacation', 'Bali travel', 'Bali hotel', 'Ubud Bali', 'Tanah Lot Bali', 'Seminyak Bali', 'Nusa Penida Bali', 'Bali surfing', 'Bali white water rafting', 'Bali food', 'best restaurants Bali', 'Bali cafe', 'Bali cooking class', 'Bali villa private pool', 'Bali resort', 'Bali boutique hotel', 'Bali airbnb', 'Bali yoga retreat', 'Bali spa massage', 'Bali meditation', 'Bali honeymoon', 'Bali waterpark', 'Bali shopping', 'Bali souvenir', 'Bali market', 'Bali fashion']

Coverage per periode:
       kw_coverage  kw_coverage_pct  low_coverage_flag
count       211.00           211.00             211.00
mean         22.43             0.86               0.03
std           3.33             0.13               0.17
min           3.00             0.12               0.00
25%          23.00             0.88               0.00
50%          23.00             0.88               0.00
75%          23.00             0.88               0.00
max          23.00             0.88   

,trend_composite,trend_risk_score,trend_rolling_mean,trend_drop_flag,kw_coverage,kw_coverage_pct,low_coverage_flag
date,,,,,,,
2008-12-01,14.652174,0.817555,NaN,0,23,0.884615,0
2009-01-01,15.739130,0.786474,15.195652,0,23,0.884615,0
2009-02-01,17.413043,0.738609,15.934783,0,23,0.884615,0
2009-03-01,18.686957,0.702182,17.279710,0,23,0.884615,0
2009-04-01,18.717391,0.701312,18.272464,0,23,0.884615,0
2009-05-01,20.443478,0.651955,19.282609,0,23,0.884615,0
2009-06-01,21.978261,0.608069,20.379710,0,23,0.884615,0
2009-07-01,22.608696,0.590042,21.676812,0,23,0.884615,0
2009-08-01,21.017391,0.635544,21.868116,0,23,0.884615,0


---
## 6. Validasi Gabungan & Merge Final

Dua langkah:
1. **Coverage check** semua file timeseries inti yang sudah disimpan ke `data/processed/` (tema 1–3).
2. **Merge** seluruh sumber risiko eksternal & digital (tema 4–5) jadi satu dataframe bulanan.


### 6.1 Coverage Check — File Processed (Tema 1–3)

In [38]:
print('=== COVERAGE VALIDATION FILE PROCESSED (TEMA PARIWISATA / EKONOMI / AKOMODASI) ===')
print()

processed = {
    'wisman_clean.csv'              : 'month',
    'wisnus_clean.csv'              : 'month',
    'monthly_usd.csv'               : 'month',
    'monthly_forex.csv'             : 'month',
    'tpk_clean.csv'                 : 'month',
    'inflasi_clean.csv'             : 'month',
    'wisman_vs_indonesia_clean.csv' : 'tahun',
    'wb_monthly_economic.csv'       : None,
    'lama_menginap_clean.csv'          : 'month',
}

for fname, date_col in processed.items():
    path = f'../data/processed/{fname}'
    if os.path.exists(path):
        df  = pd.read_csv(path)
        if date_col and date_col in df.columns:
            mn, mx = df[date_col].min(), df[date_col].max()
        else:
            mn, mx = 'N/A', 'N/A'
        mis = df.isnull().sum().sum()
        print(f'  {fname:35s} | {mn} -> {mx} | missing={mis}')
    else:
        print(f'  {fname:35s} | FILE TIDAK ADA')

print()
print('✓ Coverage check selesai.')


=== COVERAGE VALIDATION FILE PROCESSED (TEMA PARIWISATA / EKONOMI / AKOMODASI) ===

  wisman_clean.csv                    | 2009-01 -> 2026-04 | missing=0
  wisnus_clean.csv                    | 2004-01 -> 2025-12 | missing=0
  monthly_usd.csv                     | 2010-01 -> 2024-12 | missing=0
  monthly_forex.csv                   | 2014-11 -> 2026-05 | missing=0
  tpk_clean.csv                       | 2000-01 -> 2026-03 | missing=160
  inflasi_clean.csv                   | 2009-01 -> 2025-12 | missing=0
  wisman_vs_indonesia_clean.csv       | 1969 -> 2025 | missing=0
  wb_monthly_economic.csv             | N/A -> N/A | missing=0
  lama_menginap_clean.csv             | 2009-01 -> 2025-12 | missing=72

✓ Coverage check selesai.


### 6.2 Merge — Sumber Risiko Eksternal & Digital (Tema 4–5)

In [39]:
# ── Audit coverage semua sumber ───────────────────────────────────────────────
print("=== COVERAGE AUDIT ===")
print(f"gdelt.index.max()           = {gdelt.index.max().date()}")
print(f"gtrends_monthly.index.max() = {gtrends_monthly.index.max().date()}")
print(f"wb_monthly.index.max()      = {wb_monthly.index.max().date()}")
print(f"eq_monthly.index.max()      = {eq_monthly.index.max().date()}")
print(f"wx_monthly.index.max()      = {wx_monthly.index.max().date()}")
print(f"bg.index.max()              = {bg.index.max().date()}")
print()

# Drop flag imputasi sebelum merge (tetap tersimpan terpisah di wb_monthly_economic.csv untuk transparansi)
_wb_merge = wb_monthly.drop(columns=['wb_is_imputed'], errors='ignore')

start = max(gdelt.index.min(), gtrends_monthly.index.min(), _wb_merge.index.min(),
            eq_monthly.index.min(), wx_monthly.index.min(), bg.index.min())
end   = min(gdelt.index.max(), gtrends_monthly.index.max(), _wb_merge.index.max(),
            eq_monthly.index.max(), wx_monthly.index.max(), bg.index.max())

print(f"start = {start.date()}")
print(f"end   = {end.date()}")
print()

_maxes = {
    'gdelt': gdelt.index.max(), 'gtrends_monthly': gtrends_monthly.index.max(),
    'wb_monthly': _wb_merge.index.max(), 'eq_monthly': eq_monthly.index.max(),
    'wx_monthly': wx_monthly.index.max(), 'bg': bg.index.max(),
}
_bottleneck = min(_maxes, key=_maxes.get)
print(f"BOTTLENECK (sumber yang paling membatasi rentang data): {_bottleneck} = {_maxes[_bottleneck].date()}")
print()

gdelt_sel  = gdelt[["gdelt_crisis_score", "avg_tone", "risk_ratio"]].loc[start:end]
trends_sel = gtrends_monthly[["trend_composite", "trend_risk_score", "trend_drop_flag"]].loc[start:end]
wb_sel     = _wb_merge[["economic_index", "economic_risk_score"]].loc[start:end]
eq_sel     = eq_monthly[["eq_count", "eq_max_magnitude", "eq_seismic_energy", "eq_risk_score"]].loc[start:end]
wx_sel     = wx_monthly[["wx_precip_sum", "wx_humidity_mean"]].loc[start:end]
bg_sel     = bg[[
    "bmkg_max_magnitude_30d", "bmkg_seismic_energy",
    "earthquake_count", "earthquake_count_m5_plus",
    "pvmbg_volcano_status", "volcano_status_agung", "volcano_status_batur",
    "bmkg_extreme_weather_days", "weather_max_rain",
    "weather_max_wind_speed", "weather_max_wind_gust", "weather_max_temperature",
    "eq_count_log", "eq_m5plus_log", "rain_log",
    "disaster_risk_score",
]].loc[start:end]

combined = (
    gdelt_sel
    .join(trends_sel, how="outer")
    .join(wb_sel,     how="outer")
    .join(eq_sel,     how="outer")
    .join(wx_sel,     how="outer")
    .join(bg_sel,     how="outer")
)
combined = combined.sort_index()
combined = combined.interpolate(method="time", limit=2)

print(f"Shape : {combined.shape}")
print(f"Range : {combined.index.min().date()} -> {combined.index.max().date()}")
print()
print("Nulls:")
print(combined.isnull().sum())


=== COVERAGE AUDIT ===
gdelt.index.max()           = 2026-06-01
gtrends_monthly.index.max() = 2026-06-01
wb_monthly.index.max()      = 2025-12-01
eq_monthly.index.max()      = 2026-06-01
wx_monthly.index.max()      = 2026-06-01
bg.index.max()              = 2026-06-01

start = 2009-01-01
end   = 2025-12-01

BOTTLENECK (sumber yang paling membatasi rentang data): wb_monthly = 2025-12-01

Shape : (204, 30)
Range : 2009-01-01 -> 2025-12-01

Nulls:
gdelt_crisis_score           0
avg_tone                     0
risk_ratio                   0
trend_composite              0
trend_risk_score             0
trend_drop_flag              0
economic_index               0
economic_risk_score          0
eq_count                     0
eq_max_magnitude             0
eq_seismic_energy            0
eq_risk_score                0
wx_precip_sum                0
wx_humidity_mean             0
bmkg_max_magnitude_30d       0
bmkg_seismic_energy          0
earthquake_count             0
earthquake_count_m5_plus

---
## 7. Save

In [40]:
combined.to_csv("../data/processed/combined_additional_features_new.csv")
print("✓ combined_additional_features_new.csv disimpan")
print(f"  Shape : {combined.shape}")


✓ combined_additional_features_new.csv disimpan
  Shape : (204, 30)
